# Load datasets into MongoDB

Start with one JSON file and one collection. Then load train/test splits and a registered dataset.
Install BONDI with `uv sync --extra data`, start Docker, and run the cells in order.
The notebook creates `notebook-datasets` on port 27126. Change these values if they are already in use.
The final cell removes this deployment, its data and the temporary source files.
If execution stops early, run the cleanup cell manually.

See the [dataset guide](../docs/datasets.md) for file formats, external connections and Couchbase conversion rules.


In [ ]:
from pathlib import Path

from bondi.data import load_data
from bondi.deployment import deploy_mongodb

# Locate dev/business.json from the repository root or the notebooks directory.
repo_root = Path.cwd()
if not (repo_root / "dev" / "business.json").is_file():
    repo_root = repo_root.parent
business_file = repo_root / "dev" / "business.json"
assert business_file.is_file(), (
    "Run this notebook from the BONDI repository or its notebooks directory."
)

deployment = deploy_mongodb(name="notebook-datasets", port=27126)

## Load one file into one collection

Load `dev/business.json` into `dataset_demo.business`. Use each document's `business_id` as its ID.
No manifest or namespace is needed. The default mode requires an empty or absent collection.


In [ ]:
business = load_data(
    business_file,
    target=deployment,
    database="dataset_demo",
    collection="business",
    id_field="business_id",
)
print(business.destinations)
assert business.destinations == {"dataset_demo/business": 8}

## Inspect the collection

Read the businesses from MongoDB and display their IDs and names.


In [ ]:
from bondi.mongodb import MongoCollectionRef, MongoExecutor, MongoFindQuery

business_query = MongoFindQuery(MongoCollectionRef("dataset_demo", "business"), {})
with MongoExecutor.from_uri(deployment.uri) as executor:
    businesses = executor.run(business_query, collect_results=True).documents
for document in businesses:
    print(document["_id"], document["name"])
assert len(businesses) == 8

## Create a split dataset

These files represent one collection with train and test splits.
The explicit namespace keeps generated IDs stable when the files move.
Extended JSON dates become BSON dates in MongoDB.

In [ ]:
import json
from tempfile import TemporaryDirectory

from bondi.data import LoadDataError

workspace = TemporaryDirectory(prefix="bondi-dataset-demo-")
root = Path(workspace.name)
print(f"Temporary dataset directory: {root}")

train = [
    {"city": "Sydney", "visited": {"$date": "2026-01-01T00:00:00Z"}},
    {"city": "Melbourne", "visited": {"$date": "2026-01-02T00:00:00Z"}},
]
test = [{"city": "Brisbane", "visited": {"$date": "2026-01-03T00:00:00Z"}}]
for split_name, documents in {"train": train, "test": test}.items():
    (root / f"{split_name}.jsonl").write_text(
        "".join(json.dumps(document) + "\n" for document in documents)
    )
manifest = {
    "version": 1,
    "namespace": "dataset-notebook-v1",
    "collections": {"visits": {"splits": {"train": "train.jsonl", "test": "test.jsonl"}}},
}
manifest_path = root / "dataset.json"
manifest_path.write_text(json.dumps(manifest))

## Preserve splits and report progress

Each split gets its own collection. The callback receives a snapshot after each acknowledged batch.
The default write mode refuses nonempty collections.

In [ ]:
def progress(result):
    print("Acknowledged:", result.destinations)


result = load_data(
    manifest=manifest_path,
    target=deployment,
    database="dataset_demo",
    json_mode="extended",
    batch_size=2,
    on_progress=progress,
)
print("Conversions:", result.conversions)
assert result.destinations == {
    "dataset_demo/visits_train": 2,
    "dataset_demo/visits_test": 1,
}

## Inspect the loaded documents

The dates are native datetimes. Generated IDs remain in the documents.
The executor reads all returned rows.

In [ ]:
query = MongoFindQuery(MongoCollectionRef("dataset_demo", "visits_train"), {})
with MongoExecutor.from_uri(deployment.uri) as executor:
    before = executor.run(query, collect_results=True).documents
for document in before:
    print(document)

## Repeat a load explicitly

A second create load fails because the collection contains data.
Use replace to clear the selected split and load it again. The test split remains unchanged.

In [ ]:
try:
    load_data(manifest=manifest_path, target=deployment, database="dataset_demo")
except LoadDataError as error:
    print("Expected refusal:", error)

load_data(
    manifest=manifest_path,
    target=deployment,
    database="dataset_demo",
    json_mode="extended",
    split="train",
    mode="replace",
)
with MongoExecutor.from_uri(deployment.uri) as executor:
    after = executor.run(query, collect_results=True).documents
assert {d["_id"] for d in before} == {d["_id"] for d in after}
print("Generated IDs are unchanged.")

## Merge the splits

Merge loads all three documents into `visits`. It does not add a split field or change generated IDs.
The result still reports source counts for each split.

In [ ]:
merged = load_data(
    manifest=manifest_path,
    target=deployment,
    database="dataset_demo",
    json_mode="extended",
    splits="merge",
)
assert merged.destinations == {"dataset_demo/visits": 3}
print(merged.source_counts)

## Load the packaged dataset

Registered datasets provide their own namespace and source configuration.
Load this example into a separate database so it does not conflict with the single-file example.
Use `bondi datasets list` to discover other registered datasets.
Hub datasets require `bondi[data]` and network access. This example stays local.


In [ ]:
demo = load_data("demo", target=deployment, database="packaged_demo")
print(demo.destinations)
assert demo.destinations == {"packaged_demo/business": 8}

## Remove the demo resources

This deletes the demo deployment, its data volume, its saved configuration and any temporary source files.
Run this cell before restarting the notebook from an empty database.


In [ ]:
deployment.remove()
if "workspace" in globals():
    workspace.cleanup()